# Capstone Project 3 - Social Media Usage and Emotional Well-Being

In [ ]:
import json
import warnings
import joblib
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
NUMERIC_COLS = [
    'Age', 'Daily_Usage_Time (minutes)', 'Posts_Per_Day',
    'Likes_Received_Per_Day', 'Comments_Received_Per_Day',
    'Messages_Sent_Per_Day'
]
CATEGORICAL_COLS = ['Gender', 'Platform']
TARGET_COL = 'Dominant_Emotion'
VALID_GENDERS = {'Male', 'Female', 'Non-binary'}
VALID_PLATFORMS = {'Instagram', 'Twitter', 'Facebook', 'LinkedIn', 'Whatsapp', 'Telegram', 'Snapchat'}
VALID_EMOTIONS = {'Happiness', 'Anger', 'Neutral', 'Anxiety', 'Boredom', 'Sadness'}


def clean_dataframe(df):
    df = df.copy()
    junk = ['User_ID'] + [c for c in df.columns if c.startswith('Unnamed')]
    df = df.drop(columns=[c for c in junk if c in df.columns], errors='ignore')
    df = df.dropna(how='all')
    df = df[df[TARGET_COL].isin(VALID_EMOTIONS)]
    df = df[df['Gender'].isin(VALID_GENDERS)]
    df = df[df['Platform'].isin(VALID_PLATFORMS)]
    for col in NUMERIC_COLS:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=NUMERIC_COLS)
    df = df[(df['Age'] >= 10) & (df['Age'] <= 100)]
    df = df[(df['Daily_Usage_Time (minutes)'] >= 0) & (df['Daily_Usage_Time (minutes)'] <= 1440)]
    return df.reset_index(drop=True)


def build_features(df, ref_columns=None):
    X = pd.get_dummies(df[CATEGORICAL_COLS + NUMERIC_COLS], columns=CATEGORICAL_COLS, dtype=float)
    if ref_columns is not None:
        X = X.reindex(columns=ref_columns, fill_value=0)
    return X


def main():
    train_df = clean_dataframe(pd.read_csv('train.csv'))
    val_df = clean_dataframe(pd.read_csv('val.csv'))
    test_df = clean_dataframe(pd.read_csv('test.csv'))

    y_train, y_val, y_test = train_df[TARGET_COL], val_df[TARGET_COL], test_df[TARGET_COL]
    X_train_raw = build_features(train_df)
    model_columns = list(X_train_raw.columns)
    X_val_raw = build_features(val_df, model_columns)
    X_test_raw = build_features(test_df, model_columns)

    scaler = StandardScaler()
    X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=model_columns)
    X_val = pd.DataFrame(scaler.transform(X_val_raw), columns=model_columns)
    X_test = pd.DataFrame(scaler.transform(X_test_raw), columns=model_columns)

    candidates = {
        'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'Random Forest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE),
        'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
        'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=RANDOM_STATE),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=15),
        'Naive Bayes': GaussianNB(),
    }

    results = []
    for name, model in candidates.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        results.append((name, model, accuracy_score(y_val, pred), f1_score(y_val, pred, average='macro')))

    results.sort(key=lambda r: r[3], reverse=True)
    best_name, _, best_val_acc, best_val_f1 = results[0]

    # Final-training procedure: Fit the scaler on train+validation,
    # Then fit a fresh copy of the selected model on the same transformed data.
    trainval_df = pd.concat([train_df, val_df], ignore_index=True)
    y_trainval = trainval_df[TARGET_COL]
    X_trainval_raw = build_features(trainval_df, model_columns)
    final_scaler = StandardScaler()
    X_trainval = pd.DataFrame(final_scaler.fit_transform(X_trainval_raw), columns=model_columns)
    X_test_final = pd.DataFrame(final_scaler.transform(X_test_raw), columns=model_columns)

    best_template = dict(candidates)[best_name]
    final_model = best_template.__class__(**best_template.get_params())
    final_model.fit(X_trainval, y_trainval)
    test_pred = final_model.predict(X_test_final)
    test_acc = accuracy_score(y_test, test_pred)
    test_f1 = f1_score(y_test, test_pred, average='macro')

    joblib.dump(final_model, 'best_model.joblib')
    joblib.dump(final_scaler, 'scaler.joblib')
    joblib.dump(model_columns, 'model_columns.joblib')

    metadata = {
        'best_model_name': best_name,
        'validation_accuracy': best_val_acc,
        'validation_macro_f1': best_val_f1,
        'test_accuracy': test_acc,
        'test_macro_f1': test_f1,
        'classes': sorted(y_trainval.unique().tolist()),
        'numeric_cols': NUMERIC_COLS,
        'categorical_cols': CATEGORICAL_COLS,
        'model_columns': model_columns,
        'training_note': 'Final scaler and model were refit on train+validation after model selection.'
    }
    with open('model_metadata.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    print('Validation:', best_val_acc, best_val_f1)
    print('Test:', test_acc, test_f1)
    print(classification_report(y_test, test_pred))


if __name__ == '__main__':
    main()
